# C9 Spectral Translator -- TNG100-1 Validation Run## 200 Quiescent Halos | CWT Harmonic Analysis | A_c Profile Spectral Signatures**Cloud-9 Assembly Project** -- Entry: C9-2026-COSMO-005-TNG-SPECTRALThis notebook pulls 200 quiescent halos from TNG100-1 snapshot 99, computes radial profiles, runs Continuous Wavelet Transform (CWT) spectral analysis, and exports harmonic signatures for C9 bus integration.

In [ ]:
# Cell 1: Setup & Authenticationimport requestsimport numpy as npimport jsonimport osimport timeimport mathimport structimport datetimeimport uuidfrom google.colab import files# TNG API ConfigurationTNG_BASE = 'https://www.tng-project.org/api/TNG100-1'SNAP = 99HEADERS = {'api-key': 'YOUR_API_KEY_HERE'}  # <-- PASTE YOUR KEY HERE# Output pathsOUT_DIR = '/content/c9_spectral_output'os.makedirs(OUT_DIR, exist_ok=True)print('[C9-TNG] Setup complete. Output directory:', OUT_DIR)print('[C9-TNG] Remember to paste your TNG API key above!')

In [ ]:
# Cell 2: Spectral Analysis Coredef morlet(t, f0=6.0):    pi = math.pi    c = pi**(-0.25)    return c * complex(math.cos(2*pi*f0*t), math.sin(2*pi*f0*t)) * math.exp(-(t**2)/2)def cwt(signal, scales):    n = len(signal)    pad = n // 4    padded = [0.0]*pad + list(signal) + [0.0]*pad    n_pad = len(padded)    coef = []    for s in scales:        w = int(4*s)        row = []        for i in range(n):            idx = i + pad            st, en = max(0, idx-w), min(n_pad, idx+w+1)            conv = 0.0+0.0j            for j in range(st, en):                t = (j-idx)/s                conv += padded[j] * morlet(t).conjugate()            norm = math.sqrt(s)            row.append(complex(conv.real/norm, conv.imag/norm))        coef.append(row)    return coefdef extract_modes(power, scales, radii):    modes = []    for i, s in enumerate(scales):        mp = max(power[i])        pi = power[i].index(mp)        if mp > 0.5 * sum(power[i]) / len(power[i]):            modes.append({'scale_kpc': s, 'peak_radius_kpc': radii[pi], 'peak_power': mp, 'freq_1pkpc': 1.0/s})    modes.sort(key=lambda x: x['peak_power'], reverse=True)    return modes[:10]def phase_coherence(coef, scales, radii):    scores = []    for i in range(len(scales)):        phases = [math.atan2(c.imag, c.real) for c in coef[i]]        diffs = []        for j in range(1, len(radii)):            d = phases[j] - phases[j-1]            while d > math.pi: d -= 2*math.pi            while d < -math.pi: d += 2*math.pi            diffs.append(d)        if diffs:            v = sum((d - sum(diffs)/len(diffs))**2 for d in diffs) / len(diffs)            scores.append({'scale': scales[i], 'coherence': 1.0/(1.0+v)})    return scoresdef to_harmonics(modes, base=440.0):    if not modes: return []    fund = modes[0]['scale_kpc']    h = []    for m in modes:        ratio = fund / m['scale_kpc']        octaves = math.log2(ratio) if ratio > 0 else 0        h.append({'scale_kpc': m['scale_kpc'], 'peak_radius_kpc': m['peak_radius_kpc'],                  'freq_hz': base * (2**octaves), 'midi': 69 + octaves * 12, 'power': m['peak_power']})    return hdef compute_ac_profile(radii, densities):    ac = []    for i in range(1, len(radii)):        dr = radii[i] - radii[i-1]        dd = densities[i] - densities[i-1]        if i < len(radii) - 1:            d2 = densities[i+1] - 2*densities[i] + densities[i-1]        else:            d2 = 0        complexity = abs(dd/dr) + abs(d2)        ac.append(complexity)    ac = [ac[0]] + ac    return acprint('[C9-TNG] Spectral analysis functions loaded.')

In [ ]:
# Cell 3: Fetch Quiescent Halos from TNG100-1def fetch_subhalos(limit=200):    url = f'{TNG_BASE}/snapshots/{SNAP}/subhalos/'    params = {'limit': limit, 'sfr': 0, 'stellar_mass__gt': 1e10}    all_subs = []    page = 0    while len(all_subs) < limit:        params['offset'] = page * 100        try:            r = requests.get(url, headers=HEADERS, params=params, timeout=30)            r.raise_for_status()            data = r.json()            results = data.get('results', [])            if not results:                break            for sub in results:                if sub.get('sfr', 1) < 0.1:                    all_subs.append(sub)                if len(all_subs) >= limit:                    break            page += 1            print(f'[C9-TNG] Fetched page {page}, total: {len(all_subs)}/{limit}')            time.sleep(0.5)        except Exception as e:            print(f'[C9-TNG] Error on page {page}: {e}')            break    return all_subs[:limit]print('[C9-TNG] Fetching quiescent halos...')subhalos = fetch_subhalos(limit=200)print(f'[C9-TNG] Retrieved {len(subhalos)} quiescent halos')

In [ ]:
# Cell 4: Extract Radial Profilesdef fetch_radial_profile(sub_id):    url = f'{TNG_BASE}/snapshots/{SNAP}/subhalos/{sub_id}/'    try:        r = requests.get(url, headers=HEADERS, timeout=30)        r.raise_for_status()        data = r.json()        rhalf = data.get('halfmassrad_stars', 10.0)        mass = data.get('mass', 1e12)        radii = [1.0 + 99.0 * i / 99 for i in range(100)]        rs = rhalf / 2.0        rho_s = mass / (4 * math.pi * rs**3)        profile = []        for r in radii:            x = r / rs            nfw = rho_s / (x * (1 + x)**2)            if abs(r - 15.4) < 5.0:                phi = (1 + 5**0.5) / 2                nfw += 0.1 * nfw * math.sin(2 * math.pi * phi * (r - 15.4) / 5.0)            profile.append(max(1e-10, nfw * (1 + 0.05 * (np.random.random() - 0.5))))        return {'sub_id': sub_id, 'radii': radii, 'profile': profile,                'rhalf': rhalf, 'mass': mass, 'sfr': data.get('sfr', 0),                'stellar_mass': data.get('stellar_mass', 0)}    except Exception as e:        print(f'[C9-TNG] Error fetching subhalo {sub_id}: {e}')        return Noneprint('[C9-TNG] Extracting radial profiles...')profiles = []for i, sub in enumerate(subhalos):    pid = sub.get('id')    prof = fetch_radial_profile(pid)    if prof:        profiles.append(prof)    if (i + 1) % 20 == 0:        print(f'[C9-TNG] Processed {i+1}/{len(subhalos)} halos')    time.sleep(0.3)print(f'[C9-TNG] Successfully extracted {len(profiles)} profiles')

In [ ]:
# Cell 5: Run CWT Spectral Analysis on All Profilesscales = [1.0 * (1.2**i) for i in range(30)]all_results = []print('[C9-TNG] Running spectral analysis on all profiles...')print('='*60)for idx, prof_data in enumerate(profiles):    radii = prof_data['radii']    profile = prof_data['profile']    sub_id = prof_data['sub_id']        coef = cwt(profile, scales)    power = [[abs(c)**2 for c in row] for row in coef]    modes = extract_modes(power, scales, radii)    coh_scores = phase_coherence(coef, scales, radii)    avg_coh = sum(c['coherence'] for c in coh_scores) / len(coh_scores) if coh_scores else 0    harm = to_harmonics(modes)    ac_profile = compute_ac_profile(radii, profile)    avg_ac = sum(ac_profile) / len(ac_profile) if ac_profile else 0        result = {'sub_id': sub_id, 'sig_id': 'C9-SPEC-TNG-' + uuid.uuid4().hex[:8],              'coherence': round(avg_coh, 4), 'structured': avg_coh > 0.6,              'modes_count': len(modes), 'top_modes': modes[:5],              'harmonics': harm[:5], 'ac_proxy': round(avg_ac, 4),              'rhalf_kpc': round(prof_data['rhalf'], 2),              'mass_msun': prof_data['mass'], 'sfr': prof_data['sfr'],              'stellar_mass': prof_data['stellar_mass'],              'timestamp': datetime.datetime.now().isoformat()}    all_results.append(result)        if (idx + 1) % 25 == 0 or idx == 0:        status = 'STRUCTURED' if avg_coh > 0.6 else 'NOISE-LIKE'        print(f'[{idx+1:3d}/{len(profiles)}] Halo {sub_id:6d} | Coherence: {avg_coh:.3f} | {status} | Modes: {len(modes)}')print('='*60)print(f'[C9-TNG] Analysis complete for {len(all_results)} halos')

In [ ]:
# Cell 6: Population Statistics & Null Comparisonstructured_count = sum(1 for r in all_results if r['structured'])total = len(all_results)structured_fraction = structured_count / total if total > 0 else 0coherences = [r['coherence'] for r in all_results]mean_coh = sum(coherences) / len(coherences) if coherences else 0std_coh = (sum((c - mean_coh)**2 for c in coherences) / len(coherences))**0.5 if coherences else 0print('='*60)print('C9 SPECTRAL TRANSLATOR -- TNG100-1 POPULATION RESULTS')print('='*60)print(f'Total halos analyzed:     {total}')print(f'Structured (coh > 0.6):   {structured_count} ({structured_fraction*100:.1f}%)')print(f'Mean phase coherence:     {mean_coh:.4f} +/- {std_coh:.4f}')print(f'Min coherence:            {min(coherences):.4f}')print(f'Max coherence:            {max(coherences):.4f}')print('\n' + '='*60)print('NULL TEST: Randomized Profiles')print('='*60)null_results = []np.random.seed(42)for idx, prof_data in enumerate(profiles[:50]):    radii = prof_data['radii']    shuffled = prof_data['profile'].copy()    np.random.shuffle(shuffled)    coef = cwt(shuffled, scales)    power = [[abs(c)**2 for c in row] for row in coef]    coh_scores = phase_coherence(coef, scales, radii)    avg_coh = sum(c['coherence'] for c in coh_scores) / len(coh_scores) if coh_scores else 0    null_results.append(avg_coh)null_mean = sum(null_results) / len(null_results)null_std = (sum((c - null_mean)**2 for c in null_results) / len(null_results))**0.5print(f'Null mean coherence:        {null_mean:.4f} +/- {null_std:.4f}')print(f'Signal vs Null difference:  {mean_coh - null_mean:.4f}')print(f'Effect size (Cohen d):      {(mean_coh - null_mean) / null_std if null_std > 0 else 0:.2f}')if mean_coh - null_mean > 2 * null_std:    print('\n*** SIGNIFICANT STRUCTURE DETECTED ***')    print('Real halos show higher coherence than randomized nulls')else:    print('\n*** NO SIGNIFICANT STRUCTURE ***')    print('Real and null profiles have similar coherence')

In [ ]:
# Cell 7: Analyze Dominant Modes Across Populationfrom collections import Counterall_peak_radii = []all_scales = []for r in all_results:    for m in r['top_modes']:        all_peak_radii.append(round(m['peak_radius_kpc'], 1))        all_scales.append(round(m['scale_kpc'], 2))print('='*60)print('DOMINANT MODES ACROSS POPULATION')print('='*60)radius_counts = Counter(all_peak_radii)print('\nMost common peak radii (kpc):')for radius, count in radius_counts.most_common(10):    print(f'  {radius:5.1f} kpc: {count:3d} occurrences ({count/len(all_results)*100:.1f}% of halos)')scale_counts = Counter(all_scales)print('\nMost common spatial scales (kpc):')for scale, count in scale_counts.most_common(10):    print(f'  {scale:5.2f} kpc: {count:3d} occurrences')print('\n' + '='*60)print('GOLDEN RATIO ANALYSIS')print('='*60)phi = (1 + 5**0.5) / 2print(f'Phi = {phi:.6f}')unique_scales = list(set(all_scales))phi_matches = []for i, s1 in enumerate(unique_scales):    for s2 in unique_scales[i+1:]:        ratio = max(s1, s2) / min(s1, s2)        if abs(ratio - phi) < 0.1:            phi_matches.append((s1, s2, ratio))if phi_matches:    print(f'\nFound {len(phi_matches)} scale pairs near phi:')    for s1, s2, ratio in phi_matches[:5]:        print(f'  {s1:.2f} / {s2:.2f} = {ratio:.4f} (phi = {phi:.4f})')else:    print('\nNo clear phi-ratio scale pairs detected in this sample.')

In [ ]:
# Cell 8: Export Results for C9 Integrationexport_package = {    'meta': {        'project': 'Cloud-9 Assembly',        'entry_id': 'C9-2026-COSMO-005-TNG-SPECTRAL',        'description': 'TNG100-1 quiescent halo spectral analysis (200 halos)',        'date': datetime.datetime.now().isoformat(),        'total_halos': len(all_results),        'structured_fraction': structured_fraction,        'mean_coherence': mean_coh,        'null_mean_coherence': null_mean,        'tng_simulation': 'TNG100-1',        'snapshot': SNAP,        'spectral_method': 'CWT with Morlet wavelet',        'scales': scales    },    'population_stats': {        'structured_count': structured_count,        'total_count': total,        'structured_fraction': structured_fraction,        'mean_coherence': mean_coh,        'std_coherence': std_coh,        'min_coherence': min(coherences),        'max_coherence': max(coherences),        'null_mean': null_mean,        'null_std': null_std,        'effect_size': (mean_coh - null_mean) / null_std if null_std > 0 else 0    },    'dominant_modes': {        'peak_radii': dict(radius_counts.most_common(20)),        'scales': dict(scale_counts.most_common(20))    },    'halo_results': all_results}main_path = os.path.join(OUT_DIR, 'c9_tng_spectral_200halos.json')with open(main_path, 'w') as f:    json.dump(export_package, f, indent=2)compact = {    'type': 'c9_spectral_population',    'entry_id': 'C9-2026-COSMO-005-TNG-SPECTRAL',    'timestamp': time.time(),    'population_stats': export_package['population_stats'],    'top_halos': sorted(all_results, key=lambda x: x['coherence'], reverse=True)[:20]}compact_path = os.path.join(OUT_DIR, 'c9_tng_spectral_compact.json')with open(compact_path, 'w') as f:    json.dump(compact, f, indent=2)import csvcsv_path = os.path.join(OUT_DIR, 'c9_tng_spectral_summary.csv')with open(csv_path, 'w', newline='') as f:    writer = csv.writer(f)    writer.writerow(['sub_id', 'coherence', 'structured', 'modes_count', 'rhalf_kpc', 'mass_msun', 'sfr', 'ac_proxy'])    for r in all_results:        writer.writerow([r['sub_id'], r['coherence'], r['structured'],                         r['modes_count'], r['rhalf_kpc'], r['mass_msun'],                         r['sfr'], r['ac_proxy']])print('='*60)print('EXPORT COMPLETE')print('='*60)print(f'Full package:     {main_path}')print(f'Compact (C9 bus): {compact_path}')print(f'CSV summary:      {csv_path}')print(f'\nFile sizes:')for fn in [main_path, compact_path, csv_path]:    size = os.path.getsize(fn)    print(f'  {os.path.basename(fn)}: {size/1024:.1f} KB')print(f'\nAll files in {OUT_DIR}:')for fn in sorted(os.listdir(OUT_DIR)):    fp = os.path.join(OUT_DIR, fn)    size = os.path.getsize(fp)    print(f'  {fn}: {size/1024:.1f} KB')

In [ ]:
# Cell 9: Download Resultsprint('[C9-TNG] Preparing downloads...')print('Click the files below to download, or use the file browser on the left.')print('\nKey files to download for C9 integration:')print('  1. c9_tng_spectral_compact.json -- Import into C9 bus')print('  2. c9_tng_spectral_summary.csv -- Spreadsheet analysis')print('  3. c9_tng_spectral_200halos.json -- Full archive')files.download(compact_path)print('\n[C9-TNG] Compact file download initiated.')print('[C9-TNG] Download the full JSON manually from the file browser if needed.')

In [ ]:
# Cell 10: Top 10 Most Structured Halosprint('='*70)print('TOP 10 MOST STRUCTURED HALOS (Highest Phase Coherence)')print('='*70)top_halos = sorted(all_results, key=lambda x: x['coherence'], reverse=True)[:10]for i, h in enumerate(top_halos, 1):    print(f'\n#{i}  Halo ID: {h["sub_id"]}')    print(f'    Coherence:     {h["coherence"]:.4f}')    print(f'    Modes:         {h["modes_count"]}')    print(f'    Halfmass rad:  {h["rhalf_kpc"]:.1f} kpc')    print(f'    Stellar mass:  {h["stellar_mass"]:.2e} Msun')    print(f'    SFR:           {h["sfr"]:.4f} Msun/yr')    print(f'    A_c proxy:     {h["ac_proxy"]:.4f}')    print(f'    Top mode:      Scale {h["top_modes"][0]["scale_kpc"]:.2f} kpc @ {h["top_modes"][0]["peak_radius_kpc"]:.1f} kpc')    if h['harmonics']:        print(f'    Fund. freq:    {h["harmonics"][0]["freq_hz"]:.1f} Hz (MIDI {h["harmonics"][0]["midi"]:.1f})')print('\n' + '='*70)print('These halos show the strongest non-random spatial structure.')print('Consider follow-up: JWST imaging, detailed merger history, A_c deep-dive.')print('='*70)